In [40]:
import pandas as pd 
import numpy as np
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB


from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score,recall_score, f1_score

from sklearn.feature_extraction.text import TfidfVectorizer
import mlflow
import dagshub
from mlflow.models import infer_signature

In [41]:
#initialize dashub 
dagshub.init(repo_owner='amarpoji', repo_name='ai-fraud-detector', mlflow=True)


Accessing as amarpoji

Initialized MLflow to track repo "amarpoji/ai-fraud-detector"

Repository amarpoji/ai-fraud-detector initialized!

In [26]:
data_path = '../data/processed/email_dataset_processed.csv'

df = pd.read_csv(data_path)
print(df.columns)
df[['text', 'label', 'cleaned_text']]

Index(['text', 'label', 'phishing_type', 'severity', 'confidence',
       'cleaned_text'],
      dtype='object')


,text,label,cleaned_text
0,Subject: Office maintenance\n\nThanks for your...,0,subject office maintenance thanks help analysi...
1,"Hello, your profile has been locked. Use the s...",1,hello profile locked use secure link verify us...
2,"Hi there, congratulations! You are the winner ...",1,hi congratulations winner refund collect gift ...
3,"Attention, this is the fraud prevention accoun...",1,attention fraud prevention accounts team secur...
4,"Notice, your profile has been restricted. Use ...",1,notice profile restricted use secure link rese...
...,...,...,...
9995,Subject: Code review summary\n\nI booked the l...,0,subject code review summary booked lab next we...
9996,"Hello, we talked about meeting again after the...",1,hello talked meeting conference unexpected sit...
9997,"Hi there, we talked about meeting again after ...",1,hi talked meeting conference unexpected situat...
9998,"Dear user, this is an expires midnight notice ...",1,dear user expires midnight notice regarding ac...


In [42]:
X = df['cleaned_text']
y = df['label']

X_train,X_test, y_train,y_test = train_test_split(X,y, test_size = 0.2, random_state = 42, stratify = y)

# vectorize the input 
# Create and fit TF-IDF vectorizer on training data only
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)




display(X_train_vectorized)
display(X_test_vectorized)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 373355 stored elements and shape (8000, 5000)>

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 92494 stored elements and shape (2000, 5000)>

In [ ]:
mlflow.set_experiment('ai_fraud_detector_model')

with mlflow.start_run():
  

In [43]:
with mlflow.start_run(run_name="RF_GridSearch_Optimization"):
    
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=5,
        scoring='f1',
        n_jobs=-1,
        verbose=2 
    )

    # 1. Perform the search
    grid_search.fit(X_train_vectorized, y_train)

    # 2. EXTRACT THE WINNER (Crucial!)
    best_model = grid_search.best_estimator_

    # 3. LOG THE BEST PARAMETERS (Crucial!)
    mlflow.log_params(grid_search.best_params_)

    # 4. Log CV Metric
    mlflow.log_metric("cv_best_f1", grid_search.best_score_)

    # 5. Evaluate on Test Set
    # Make sure X_test is also vectorized!
    y_pred = best_model.predict(X_test_vectorized)
    
    test_f1 = f1_score(y_test, y_pred)
    test_acc = accuracy_score(y_test, y_pred)
        
    mlflow.log_metrics({
        "test_f1_score": test_f1,
        "test_accuracy": test_acc
    })

    # 6. Signature (Schema)
    # We use a tiny sample and convert to dense only for the schema
    sample_input = X_train_vectorized[:3].toarray() 
    signature = infer_signature(sample_input, best_model.predict(sample_input))
        
    # 7. Log Model
    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
        signature=signature,
        registered_model_name="Phishing_Classifier_RF"
    )

    print(f"Successfully logged best model with Test F1: {test_f1}")

Fitting 5 folds for each of 54 candidates, totalling 270 fits


2026/04/07 13:29:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/07 13:29:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Phishing_Classifier_RF'.
2026/04/07 13:29:30 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Phishing_Classifier_RF, version 1
Created version '1' of model 'Phishing_Classifier_RF'.


Successfully logged best model with Test F1: 1.0
🏃 View run RF_GridSearch_Optimization at: https://dagshub.com/amarpoji/ai-fraud-detector.mlflow/#/experiments/1/runs/06ffd47b71574cff8709296b2bf4ba20
🧪 View experiment at: https://dagshub.com/amarpoji/ai-fraud-detector.mlflow/#/experiments/1


In [ ]:
# Execute Tuning
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_

        # 6. Log Best Parameters & CV Metrics
        mlflow.log_params(grid_search.best_params_)
        mlflow.log_metric("cv_best_f1", grid_search.best_score_)

        # 7. Evaluate on Test Set
        y_pred = best_model.predict(X_test)
        test_f1 = f1_score(y_test, y_pred)
        test_acc = accuracy_score(y_test, y_pred)
        
        mlflow.log_metrics({
            "test_f1_score": test_f1,
            "test_accuracy": test_acc
        })

        print(f"✅ Best CV F1: {grid_search.best_score_:.4f}")
        print(f"📊 Test F1: {test_f1:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Use the best model to predict on new data
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_vectorized)

# Calculate metrics
metrics = {
    "test_accuracy": accuracy_score(y_test, y_pred),
    "test_precision": precision_score(y_test, y_pred),
    "test_recall": recall_score(y_test, y_pred),
    "test_f1": f1_score(y_test, y_pred)
}

# Log them all at once
mlflow.log_metrics(metrics)